In [31]:
import pandas as pd
import numpy as np

from operator import itemgetter

from common.county_geometry import convert_points
from common.county_geometry import central_angle, earth_radius_mi

data_path = "./data/cleaner/centerlines_clean.json"

def read_in_centerlines(path_to_data):
    df = pd.read_json(path_to_data, orient='records', lines=True)
    return df.set_index("OBJECTID")

centerlines = read_in_centerlines(data_path)
centerlines.head()

,ROADNAME,SIFID,SIFCODE,low_cross_ROADNAME,SIFIDLOW,LOCROSSSIF,hi_cross_ROADNAME,SIFIDHI,HICROSSSIF,CORE_CLASS,GEOMETRY
OBJECTID,,,,,,,,,,,
1,SERENITY CT,8665,9887,DELLAFAY DR,1550,1557,DEAD END,8594,9811,LOCAL,"[[-85.6809503218278, 38.158867088749105], [-85..."
2,S 28TH ST,5926,6570,W HILL ST,2854,2934,W GAULBERT AVE,2470,2499,LOCAL,"[[-85.80120122370631, 38.23056379303306], [-85..."
3,BEECH ST,473,0458,WILSON AVE,6487,7212,DR WILLIAM G WEATHERS DR,10596,D596,LOCAL,"[[-85.80501498815028, 38.228933021550915], [-8..."
4,GARDEN DR,2442,2470,RAINBOW DR,13394,5391,POPPY WAY,4702,5128,PRIMARY COLLECTOR,"[[-85.68020545343364, 38.24806713661984], [-85..."
5,PARKWAY DR,4573,4974,MOUNT CLAIRE AVE,4162,4476,DEAD END,8594,9811,LOCAL,"[[-85.74203979766622, 38.211814770925905], [-8..."


In [40]:
GEO_ends = centerlines.GEOMETRY.transform({'GEO_start':itemgetter(0), "GEO_end":itemgetter(-1)})
geo_ends_LL = GEO_ends
geo_ends_LL

LL_segment_distance = geo_ends_LL.GEO_start.combine(geo_ends_LL.GEO_end, central_angle)
LL_segment_distance = LL_segment_distance*earth_radius_mi*5280 # convert to feet

In [46]:
geo_ends_KYgrid = GEO_ends.map(convert_points.point_to_grid)
geo_ends_KYgrid

def grid_distance(grid1, grid2):
    gx1, gy1 = grid1
    gx2, gy2 = grid2
    dx = gx1 - gx2
    dy = gy1 - gy2 
    return np.sqrt(dx**2 + dy**2) # ft value

KYgrid_segment_distance = geo_ends_KYgrid.GEO_start.combine(geo_ends_KYgrid.GEO_end, grid_distance)

diff = (LL_segment_distance - KYgrid_segment_distance).abs() # ft
diff_in = diff * 12 # convert to inches

diff_in[diff_in >= 36]


OBJECTID
19        50.561000
25        37.013293
40        44.084832
57        85.930817
62        52.976277
            ...    
177348    52.828739
178627    40.463643
178945    40.442294
178958    44.236714
180227    68.353166
Length: 1511, dtype: float64

In [ ]:
(diff_in/12).describe()
# bigger difference in conversion

# Conclusion
# use longitude, latitude for distance comparisons.

count    35039.000000
mean         0.906686
std          1.571212
min          0.000000
25%          0.236379
50%          0.528665
75%          1.085232
max         52.542465
dtype: float64